# 📊 01 - Khám phá dữ liệu (EDA)

Notebook này phân tích dataset MC-OCR 2021 để hiểu đặc điểm dữ liệu trước khi huấn luyện.

**Nội dung:**
1. Setup môi trường
2. Load và kiểm tra cấu trúc CSV
3. Phân tích tính toàn vẹn dữ liệu
4. Phân bố chất lượng ảnh
5. Phân bố nhãn entity
6. Thống kê số dòng text mỗi ảnh
7. Hiển thị ảnh mẫu kèm bounding box
8. Tóm tắt phát hiện

## 1. Setup môi trường

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repo (nếu chưa có)
import os

REPO_DIR = '/content/document-ai-vn'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Duc-AnhTp/document-ai-vn.git {REPO_DIR}
else:
    print(f'Repo đã tồn tại tại {REPO_DIR}')

os.chdir(REPO_DIR)

import sys
sys.path.insert(0, REPO_DIR)

In [ ]:
# Import thư viện
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from pathlib import Path
from PIL import Image

from configs.paths import CSV_PATH, IMAGE_DIR
from src.preprocessing.parse_mcocr import parse_mcocr_csv, safe_split, safe_literal_eval

print(f'CSV path: {CSV_PATH}')
print(f'Image dir: {IMAGE_DIR}')
print(f'CSV exists: {CSV_PATH.exists()}')
print(f'Image dir exists: {IMAGE_DIR.exists()}')

## 2. Load và kiểm tra cấu trúc CSV

In [ ]:
df = pd.read_csv(CSV_PATH)

print(f'Số dòng: {len(df)}')
print(f'Các cột: {list(df.columns)}')
print()
df.head(3)

In [ ]:
# Thống kê cơ bản
print('=== Thống kê kiểu dữ liệu ===')
print(df.dtypes)
print()
print('=== Giá trị null ===')
print(df.isnull().sum())
print()
print('=== Mẫu dữ liệu annotation ===')
sample_row = df.iloc[0]
print(f"img_id: {sample_row.get('img_id', 'N/A')}")
print(f"anno_texts (raw): {str(sample_row.get('anno_texts', ''))[:200]}")
print(f"anno_labels (raw): {str(sample_row.get('anno_labels', ''))[:200]}")

## 3. Phân tích tính toàn vẹn dữ liệu

In [ ]:
# Parse toàn bộ dataset
samples = parse_mcocr_csv(str(CSV_PATH))
print(f'Tổng số sample parse được: {len(samples)}')

In [ ]:
# Kiểm tra ảnh thiếu
missing_images = []
existing_images = []
for sample in samples:
    img_path = IMAGE_DIR / sample['image_id']
    if img_path.exists():
        existing_images.append(sample)
    else:
        missing_images.append(sample['image_id'])

print(f'Ảnh tồn tại: {len(existing_images)}')
print(f'Ảnh thiếu: {len(missing_images)}')
if missing_images:
    print(f'Ví dụ ảnh thiếu: {missing_images[:5]}')

In [ ]:
# Kiểm tra text-label lệch
mismatch_count = 0
no_text_count = 0
no_label_count = 0

for sample in samples:
    texts = sample.get('texts', [])
    labels = sample.get('labels', [])

    if not texts:
        no_text_count += 1
    if not labels:
        no_label_count += 1
    if len(texts) != len(labels) and texts and labels:
        mismatch_count += 1

print(f'Sample không có text: {no_text_count}')
print(f'Sample không có label: {no_label_count}')
print(f'Sample text-label lệch số lượng: {mismatch_count}')

## 4. Phân bố chất lượng ảnh

In [ ]:
from src.visualization.plot_samples import plot_quality_distribution

plot_quality_distribution(samples)

# Thống kê thêm
qualities = [float(s['image_quality']) for s in samples 
             if s.get('image_quality') is not None]
if qualities:
    print(f'Min quality: {min(qualities):.4f}')
    print(f'Max quality: {max(qualities):.4f}')
    print(f'Mean quality: {np.mean(qualities):.4f}')
    print(f'Median quality: {np.median(qualities):.4f}')
    
    low = sum(1 for q in qualities if q < 0.33)
    medium = sum(1 for q in qualities if 0.33 <= q < 0.66)
    high = sum(1 for q in qualities if q >= 0.66)
    print(f'\nLow (<0.33): {low} ({100*low/len(qualities):.1f}%)')
    print(f'Medium (0.33-0.66): {medium} ({100*medium/len(qualities):.1f}%)')
    print(f'High (>=0.66): {high} ({100*high/len(qualities):.1f}%)')

## 5. Phân bố nhãn entity

In [ ]:
from src.visualization.plot_samples import plot_label_distribution

plot_label_distribution(samples)

# Thống kê chi tiết
all_labels = []
for sample in samples:
    all_labels.extend(sample.get('labels', []))

label_counts = Counter(all_labels)
print('\nChi tiết phân bố nhãn:')
for label, count in label_counts.most_common():
    print(f'  {label}: {count} ({100*count/len(all_labels):.1f}%)')

## 6. Thống kê số dòng text mỗi ảnh

In [ ]:
text_counts = [len(s.get('texts', [])) for s in samples]

plt.figure(figsize=(10, 5))
plt.hist(text_counts, bins=30, color='#6C5CE7', edgecolor='#2C3E50', alpha=0.8)
plt.xlabel('Số dòng text mỗi ảnh')
plt.ylabel('Số lượng ảnh')
plt.title('Phân bố số dòng text mỗi ảnh')
plt.axvline(x=np.mean(text_counts), color='red', linestyle='--', label=f'Mean={np.mean(text_counts):.1f}')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Min: {min(text_counts)}')
print(f'Max: {max(text_counts)}')
print(f'Mean: {np.mean(text_counts):.1f}')
print(f'Median: {np.median(text_counts):.1f}')

## 7. Hiển thị ảnh mẫu kèm bounding box

In [ ]:
from src.visualization.plot_samples import plot_sample_with_boxes
from src.utils.bbox import polygon_to_bbox

# Lấy 5 sample có ảnh tồn tại và có polygon
display_samples = [
    s for s in samples
    if (IMAGE_DIR / s['image_id']).exists() and s.get('polygons')
][:5]

for sample in display_samples:
    image_path = str(IMAGE_DIR / sample['image_id'])
    boxes = [polygon_to_bbox(p) for p in sample['polygons']]
    labels = sample.get('labels', [])
    
    # Cắt labels theo số lượng boxes
    min_len = min(len(boxes), len(labels))
    plot_sample_with_boxes(
        image_path,
        boxes[:min_len],
        labels[:min_len],
        title=sample['image_id'],
    )

## 8. Tóm tắt phát hiện

Ghi chú kết quả EDA tại đây sau khi chạy:

- **Tổng số mẫu**: ...
- **Ảnh thiếu**: ...
- **Chất lượng ảnh**: phần lớn ... (low/medium/high)
- **Nhãn phổ biến nhất**: ...
- **Trung bình số dòng text/ảnh**: ...
- **Vấn đề dữ liệu phát hiện**: ...